# Distill LoRA → LCM-LoRA (1-step student)

Take the LoRA produced by `train_catvton_lora_vitonhd.ipynb` and distill it into an LCM-LoRA that converges in 1-2 inference steps instead of 8.

**Goal**: 8 steps × 100 ms → 1 step × 100 ms = **~8× faster inference**.

**Method**: Latent-Consistency-Model self-distillation. The student LoRA learns to produce the same final output as the teacher (the LoRA you already trained) but in a single denoising step. Reference: https://huggingface.co/docs/diffusers/training/lcm_distill

**Setup**: Same Kaggle notebook environment as training. 2× T4 or 1× P100. Free.

**Input**: `lucy_catvton_lora.safetensors` from Stage 2.  
**Output**: `lucy_catvton_lcm_lora.safetensors` — load this with the teacher LoRA at inference.

**Wall time**: ~6–10 hours on Kaggle T4×2 (3 epochs × ~3 hrs each). Fits in one Kaggle session (12 hr max) if you don't waste setup time.

## Cell 1 — Setup

In [ ]:
!pip install -q diffusers==0.30.0 transformers==4.44.0 accelerate==0.33.0 \
                peft==0.12.0 safetensors==0.4.5 wandb==0.17.7

import os, math, gc
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from accelerate import Accelerator
from diffusers import (
    StableDiffusionInpaintPipeline, UNet2DConditionModel,
    DDPMScheduler, LCMScheduler, AutoencoderKL,
)
from peft import LoraConfig, get_peft_model, PeftModel
from transformers import CLIPTextModel, CLIPTokenizer
from safetensors.torch import save_file

BASE_MODEL = 'runwayml/stable-diffusion-inpainting'
TEACHER_LORA_PATH = '/kaggle/working/lucy_catvton_lora.safetensors'   # from Stage 2
OUTPUT_DIR = '/kaggle/working/lcm_distill_output'
DATA_ROOT = '/kaggle/input/viton-hd-resized'                          # same dataset as Stage 2
RESOLUTION = 512
BATCH_SIZE = 4
LR = 1e-4
EPOCHS = 3
NUM_TIMESTEPS_STUDENT = 4   # student noise-schedule subdivisions (LCM target)
GUIDANCE_TEACHER = 2.5

os.makedirs(OUTPUT_DIR, exist_ok=True)
torch.backends.cuda.matmul.allow_tf32 = True
print(f'CUDA available: {torch.cuda.is_available()}, devices: {torch.cuda.device_count()}')

## Cell 2 — Reuse the VITON-HD dataset class

Paste the same `VitonHDDataset` you used in `train_catvton_lora_vitonhd.ipynb`. It loads (person, garment, masked_person, target_result) tuples.

In [ ]:
# Re-import from the teacher notebook OR paste the class here verbatim.
# Assumed contract: returns dict with keys 'person', 'garment', 'masked_person', 'mask', 'result' as torch tensors in [-1, 1].
from train_catvton_lora_vitonhd import VitonHDDataset  # adjust import if pasted inline

train_ds = VitonHDDataset(root=DATA_ROOT, split='train', size=RESOLUTION)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
print(f'Train set: {len(train_ds)} samples, {len(train_loader)} batches')

## Cell 3 — Load teacher (8-step) and student (LCM)

Teacher = SD-inpaint + your trained LoRA, runs 8 DDPM steps with CFG.  
Student = SD-inpaint + a fresh LCM-LoRA on top of the same UNet, runs 1 step without CFG.

In [ ]:
dtype = torch.float16
device = 'cuda'

# Teacher
teacher = StableDiffusionInpaintPipeline.from_pretrained(
    BASE_MODEL, torch_dtype=dtype, safety_checker=None, requires_safety_checker=False,
).to(device)
teacher.load_lora_weights(TEACHER_LORA_PATH)
teacher.fuse_lora()
teacher.unet.requires_grad_(False)
teacher_scheduler = DDPMScheduler.from_pretrained(BASE_MODEL, subfolder='scheduler')
teacher_scheduler.set_timesteps(8)

# Student UNet = clone of fused-LoRA UNet, then add a NEW small LoRA on top.
# This new LoRA is what we'll train + save as the LCM-LoRA.
student_unet = UNet2DConditionModel.from_pretrained(
    BASE_MODEL, subfolder='unet', torch_dtype=dtype,
    in_channels=9, ignore_mismatched_sizes=True,
).to(device)
# Apply the teacher LoRA, then freeze, then attach a fresh LCM-LoRA.
from peft import set_peft_model_state_dict
lcm_lora_config = LoraConfig(
    r=64,
    lora_alpha=64,
    init_lora_weights='gaussian',
    target_modules=['to_q', 'to_k', 'to_v', 'to_out.0'],
)
student_unet = get_peft_model(student_unet, lcm_lora_config)
student_unet.print_trainable_parameters()

vae = AutoencoderKL.from_pretrained(BASE_MODEL, subfolder='vae', torch_dtype=dtype).to(device).eval()
vae.requires_grad_(False)
tokenizer = CLIPTokenizer.from_pretrained(BASE_MODEL, subfolder='tokenizer')
text_encoder = CLIPTextModel.from_pretrained(BASE_MODEL, subfolder='text_encoder', torch_dtype=dtype).to(device).eval()
text_encoder.requires_grad_(False)
with torch.no_grad():
    null_ids = tokenizer([''], padding='max_length', max_length=77, truncation=True, return_tensors='pt').input_ids.to(device)
    null_emb = text_encoder(null_ids)[0]
print('Teacher + student loaded.')

## Cell 4 — LCM distillation loss

Core idea: for each batch we pick two adjacent timesteps `t > s`. The teacher denoises from `t → s` over many sub-steps; the student is forced to make the same jump in **one** step. We minimise the MSE between teacher's predicted `x_s` and student's predicted `x_s`.

Reference: "Latent Consistency Models" §3.3, plus the HF `train_lcm_distill_lora.py` script. We replicate the core loss with inpaint-specific 9-channel concat.

In [ ]:
def get_concat_latents(person, masked_person, mask):
    """Inpaint UNet wants [noisy_lat(4), mask_lat(1), masked_lat(4)] = 9 channels."""
    with torch.no_grad():
        masked_lat = vae.encode(masked_person).latent_dist.sample() * vae.config.scaling_factor
    mask_lat = F.interpolate(mask, size=masked_lat.shape[-2:], mode='nearest')
    return masked_lat, mask_lat

def teacher_predict_x_s(noisy_lat_t, t, s, masked_lat, mask_lat, prompt_emb):
    """Run the teacher from t down to s with its 8-step schedule."""
    x = noisy_lat_t.clone()
    sub_timesteps = [ts for ts in teacher_scheduler.timesteps if t.item() >= ts.item() >= s.item()]
    for ts in sub_timesteps:
        inp = torch.cat([x, mask_lat, masked_lat], dim=1).to(dtype)
        with torch.no_grad():
            noise_pred = teacher.unet(inp, ts.to(device), encoder_hidden_states=prompt_emb).sample
        x = teacher_scheduler.step(noise_pred, ts, x).prev_sample
    return x

def student_predict_x_s(noisy_lat_t, t, masked_lat, mask_lat, prompt_emb):
    """Student does a single-step jump."""
    inp = torch.cat([noisy_lat_t, mask_lat, masked_lat], dim=1).to(dtype)
    noise_pred = student_unet(inp, t.to(device), encoder_hidden_states=prompt_emb).sample
    # Convert noise pred → x0 prediction (LCM uses x0-prediction)
    alpha_prod_t = teacher_scheduler.alphas_cumprod[t.long()].to(noisy_lat_t.device).view(-1, 1, 1, 1)
    x0_pred = (noisy_lat_t - (1 - alpha_prod_t).sqrt() * noise_pred) / alpha_prod_t.sqrt()
    return x0_pred

print('LCM helpers ready.')

## Cell 5 — Training loop

In [ ]:
accelerator = Accelerator(mixed_precision='fp16')
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, student_unet.parameters()), lr=LR)
student_unet, optimizer, train_loader = accelerator.prepare(student_unet, optimizer, train_loader)

global_step = 0
for epoch in range(EPOCHS):
    for batch in train_loader:
        person      = batch['result'].to(device, dtype)    # target = clean image
        masked      = batch['masked_person'].to(device, dtype)
        mask        = batch['mask'].to(device, dtype)

        # Encode target → clean latent x0
        with torch.no_grad():
            x0 = vae.encode(person).latent_dist.sample() * vae.config.scaling_factor
        masked_lat, mask_lat = get_concat_latents(person, masked, mask)

        # Sample (t, s) with t > s, s either 0 or earlier teacher timestep
        all_ts = teacher_scheduler.timesteps.to(device)
        i_t = torch.randint(0, len(all_ts) - 1, (x0.shape[0],), device=device)
        i_s = i_t + 1   # adjacent step (one teacher-sub-step apart)
        t = all_ts[i_t]
        s = all_ts[i_s.clamp(max=len(all_ts)-1)]

        # Forward noise to t
        noise = torch.randn_like(x0)
        x_t = teacher_scheduler.add_noise(x0, noise, t)

        # Teacher target (no_grad): denoise from t → s
        with torch.no_grad():
            teacher_x_s = teacher_predict_x_s(x_t, t, s, masked_lat, mask_lat, null_emb.expand(x0.shape[0], -1, -1))
            # Convert teacher's x_s → x0 estimate using the same formula
            alpha_s = teacher_scheduler.alphas_cumprod[s.long()].view(-1, 1, 1, 1).to(device)
            teacher_x0_target = teacher_x_s   # at single-step distillation we compare x0 estimates directly

        # Student prediction (with grad)
        student_x0 = student_predict_x_s(x_t, t, masked_lat, mask_lat, null_emb.expand(x0.shape[0], -1, -1))

        loss = F.mse_loss(student_x0.float(), teacher_x0_target.float())

        accelerator.backward(loss)
        optimizer.step()
        optimizer.zero_grad()

        if global_step % 50 == 0:
            print(f'epoch {epoch} step {global_step} loss {loss.item():.4f}')
        global_step += 1

    # Save checkpoint each epoch — Kaggle 12 hr session means we must be resumable
    ckpt_path = os.path.join(OUTPUT_DIR, f'lcm_lora_epoch{epoch}.safetensors')
    student_unet.save_pretrained(os.path.join(OUTPUT_DIR, f'epoch{epoch}'))
    print(f'saved epoch {epoch} checkpoint')

print('LCM distillation complete.')

## Cell 6 — Export final LCM-LoRA

Merge the PEFT state-dict into a single safetensors that the inference server can load with `pipeline.load_lora_weights()`.

In [ ]:
FINAL_PATH = '/kaggle/working/lucy_catvton_lcm_lora.safetensors'
# Extract LoRA weights only (the deltas, not the full UNet)
lora_state = {k: v.detach().cpu().contiguous() for k, v in student_unet.state_dict().items() if 'lora_' in k}
save_file(lora_state, FINAL_PATH)
print(f'Wrote {FINAL_PATH}  ({len(lora_state)} tensors)')

## Cell 7 — Sanity-check at inference time

Load fresh pipeline, attach teacher + student LoRAs, run a 1-step inpaint, compare to 8-step teacher.

In [ ]:
from diffusers import LCMScheduler

pipe = StableDiffusionInpaintPipeline.from_pretrained(
    BASE_MODEL, torch_dtype=dtype, safety_checker=None, requires_safety_checker=False,
).to(device)
pipe.load_lora_weights(TEACHER_LORA_PATH, adapter_name='teacher')
pipe.load_lora_weights(FINAL_PATH, adapter_name='lcm')
pipe.set_adapters(['teacher', 'lcm'], adapter_weights=[1.0, 1.0])
pipe.scheduler = LCMScheduler.from_config(pipe.scheduler.config)

from PIL import Image
import random
sample = train_ds[random.randrange(len(train_ds))]
result = pipe(
    prompt='photo of a person wearing the garment, sharp focus',
    image=Image.fromarray(((sample['masked_person'].permute(1, 2, 0).numpy() + 1) * 127.5).astype('uint8')),
    mask_image=Image.fromarray((sample['mask'][0].numpy() * 255).astype('uint8')),
    num_inference_steps=1,
    guidance_scale=1.0,
).images[0]
display(result)

## Download artifact

Download `lucy_catvton_lcm_lora.safetensors` from the Kaggle output panel. This goes to the server alongside the teacher LoRA — Tier 2 path in `tryon_backend/model.py` loads both.